# Mineração de texto - recursos introdutórios

In [5]:
!pip install -r ../requirements.txt

  Using cached python_youtube-0.9.8-py3-none-any.whl.metadata (8.0 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached beautifulsoup4-4.13.5-py3-none-any.whl.metadata (3.8 kB)
  Using cached selenium-4.35.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached trio-0.30.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached websocket_client-1.8.0-py3-none-any.

In [6]:
import pandas as pd
import re
import nltk
import spacy
import string
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
## Garantir que tenha pelo menos esses dados:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

#Você pode instalar tudo disponível executando essa linha
#nltk.download()

[nltk_data] Downloading package stopwords to /Users/pedro/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/pedro/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/pedro/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Preprocessamento básico com NTLK 

#### Separação de sentenças

In [8]:
from nltk.tokenize import sent_tokenize

texto = "A fazenda é muito grande. O Dr. da Fazenda possui vários animais. Aqui temos um olho d'água."

sentencas = sent_tokenize(texto)

for sent in sentencas:
    print(sent)

A fazenda é muito grande.
O Dr. da Fazenda possui vários animais.
Aqui temos um olho d'água.


#### Tokenização

In [9]:
from nltk.tokenize import word_tokenize

texto = "A fazenda é muito grande. O Dr. da Fazenda possui vários animais. Aqui temos um olho d'água."

palavras = word_tokenize(texto) 

for palavra in palavras:
    print(palavra)

A
fazenda
é
muito
grande
.
O
Dr.
da
Fazenda
possui
vários
animais
.
Aqui
temos
um
olho
d'água
.


### Exemplificando com dados de reviews do Yelp

In [10]:
dfCharlotte= pd.read_csv("../data/reviewsTestCharlotte.csv")

reviewsCharlotte_original = dfCharlotte['text']
reviewsCharlotte_original

reviewsCharlotte_original[0]

'I loved the customer service! Fast, but the doctor took her time to make sure I was happy in my contact lenses. Thank you!'

In [11]:
reviewsCharlotte_original.shape

(48617,)

#### Convertendo para minúsculo

In [12]:
reviewsCharlotte_prep = reviewsCharlotte_original.str.lower()
reviewsCharlotte_prep

0        i loved the customer service! fast, but the do...
1        i've been coming here for a couple of years bu...
2        i really liked the optometrist here. the wait ...
3        robert and his team took great care of me when...
4        edit: i'm lowering this review from 5-stars to...
                               ...                        
48612    we've now had mike out twice to fix some leaks...
48613    love the new location lauren is always right o...
48614    i completely ruined my hair today attempting t...
48615    i have been going to lauren for years now and ...
48616    always available when i need her, lauren rocks...
Name: text, Length: 48617, dtype: object

#### Remove pontuação

In [13]:
PONTUACAO = string.punctuation

def remove_pontuacao(text):
    return text.translate(str.maketrans('', '', PONTUACAO))

reviewsCharlotte_prep = reviewsCharlotte_prep.apply(lambda text: remove_pontuacao(text))
reviewsCharlotte_prep

0        i loved the customer service fast but the doct...
1        ive been coming here for a couple of years but...
2        i really liked the optometrist here the wait w...
3        robert and his team took great care of me when...
4        edit im lowering this review from 5stars to 3s...
                               ...                        
48612    weve now had mike out twice to fix some leaks ...
48613    love the new location lauren is always right o...
48614    i completely ruined my hair today attempting t...
48615    i have been going to lauren for years now and ...
48616    always available when i need her lauren rocks ...
Name: text, Length: 48617, dtype: object

#### Remove stopwords

In [14]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])

reviewsCharlotte_prep = reviewsCharlotte_prep.apply(lambda text: remove_stopwords(text))
print(reviewsCharlotte_prep)

print(STOPWORDS)


0        loved customer service fast doctor took time m...
1        ive coming couple years optometrist business b...
2        really liked optometrist wait little longer wo...
3        robert team took great care repairing car back...
4        edit im lowering review 5stars 3stars wrote or...
                               ...                        
48612    weve mike twice fix leaks pex pipes first time...
48613    love new location lauren always right point lo...
48614    completely ruined hair today attempting highli...
48615    going lauren years would never see another hai...
48616    always available need lauren rocks particular ...
Name: text, Length: 48617, dtype: object
{'needn', 'themselves', 'most', 'between', 'and', 'but', 'just', 'is', 'was', 't', 'has', 'where', 'such', 'been', 'very', 'any', 'before', 'm', 'as', "aren't", 'o', "we'd", 'yourselves', 'myself', 'don', 'which', 'what', 'whom', 'same', "haven't", 'of', 'while', "hasn't", "don't", 'over', "i'll", 'does', 'it

#### Stemming

In [16]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])

reviewsCharlotte_prep= reviewsCharlotte_prep.apply(lambda text: stem_words(text))
reviewsCharlotte_prep

KeyboardInterrupt: 

# Topic Modeling

In [17]:
!pip install gensim

  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached scipy-1.13.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (60 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
Using cached scipy-1.13.1-cp312-cp312-macosx_12_0_arm64.whl (30.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.3
    Uninstalling numpy-2.3.3:
      Successfully uninstalled numpy-2.3.3
  Attempting uninstall: scipy━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: scipy 1.16.20m━━━━━━━━━━━━━━━━━━━ 1/2 [scipy]
    Uninstalling scipy-1.16.2:90m╺━━━━━━━━━━━━━━━━━━━ 1/2 [scipy]
      Successfully uninstalled scipy-1.16.2━━━━━━━━━━━━━━━━━━━ 1/2 [scipy]
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [scipy]^C


In [18]:
from gensim.models.ldamodel import LdaModel
from gensim.corpora.dictionary import Dictionary

### Sem pré-processamento

In [19]:
dataset_orig = [d.split() for d in reviewsCharlotte_original]
dictionary_orig = Dictionary(dataset_orig)
corpus_orig = [dictionary_orig.doc2bow(doc) for doc in dataset_orig]

model_original =LdaModel(corpus=corpus_orig, id2word=dictionary_orig, num_topics=10, iterations=100, passes=5,random_state=1)


In [20]:
for i in  model_original.show_topics(num_topics=10, num_words=6, log=False):
    print(i)
    print('---')


(0, '0.023*"games" + 0.010*"cookies" + 0.009*"Asked" + 0.005*"Rooms" + 0.005*"delivered." + 0.004*"purpose"')
---
(1, '0.040*"I" + 0.036*"to" + 0.033*"the" + 0.020*"a" + 0.017*"you" + 0.016*"that"')
---
(2, '0.021*"massage" + 0.008*"Taste" + 0.006*"sake" + 0.006*"Still," + 0.005*"Suites" + 0.005*"Avoid"')
---
(3, '0.005*"Ice" + 0.004*"mold" + 0.004*"Awful" + 0.003*"pleasure." + 0.003*"this?" + 0.003*"Cesar"')
---
(4, '0.050*"the" + 0.040*"and" + 0.039*"was" + 0.030*"I" + 0.026*"a" + 0.016*"The"')
---
(5, '0.015*"Thai" + 0.015*"ribs" + 0.008*"tire" + 0.007*"pork," + 0.007*"Taco" + 0.006*"Fantastic"')
---
(6, '0.012*"Hall" + 0.010*"Western" + 0.004*"tipped" + 0.004*"Yeah," + 0.003*"watch." + 0.002*":)."')
---
(7, '0.045*"hair" + 0.020*"salon" + 0.011*"cut" + 0.010*"color" + 0.010*"polish" + 0.008*"pedicure"')
---
(8, '0.046*"the" + 0.043*"and" + 0.039*"a" + 0.029*"is" + 0.023*"to" + 0.021*"of"')
---
(9, '0.046*"the" + 0.044*"and" + 0.041*"to" + 0.040*"I" + 0.038*"was" + 0.022*"a"')
---


### Com pré-processamento

In [21]:
dataset_prep = [d.split() for d in reviewsCharlotte_prep]
dictionary_prep = Dictionary(dataset_prep)
corpus_prep = [dictionary_prep.doc2bow(doc) for doc in dataset_prep]

model_prep =LdaModel(corpus=corpus_prep, id2word=dictionary_prep, num_topics=10, iterations=100, passes=5,random_state=1)


In [32]:
for i in  model_prep.show_topics(num_topics=10, num_words=6, log=False):
    print(i)
    print('---')


(0, '0.024*"beer" + 0.014*"bar" + 0.013*"place" + 0.013*"great" + 0.012*"good" + 0.010*"nice"')
---
(1, '0.023*"order" + 0.023*"food" + 0.015*"time" + 0.015*"wait" + 0.014*"us" + 0.013*"minut"')
---
(2, '0.021*"breakfast" + 0.018*"coffe" + 0.017*"egg" + 0.017*"brunch" + 0.014*"ice" + 0.013*"cream"')
---
(3, '0.044*"great" + 0.035*"food" + 0.031*"place" + 0.025*"servic" + 0.019*"good" + 0.018*"love"')
---
(4, '0.022*"call" + 0.016*"would" + 0.012*"told" + 0.011*"custom" + 0.011*"said" + 0.010*"servic"')
---
(5, '0.043*"sub" + 0.027*"gift" + 0.023*"japanes" + 0.022*"groceri" + 0.021*"jerk" + 0.018*"fuel"')
---
(6, '0.020*"good" + 0.019*"burger" + 0.015*"chicken" + 0.014*"fri" + 0.014*"order" + 0.011*"like"')
---
(7, '0.016*"like" + 0.014*"get" + 0.012*"go" + 0.010*"im" + 0.010*"look" + 0.010*"place"')
---
(8, '0.123*"airport" + 0.083*"room" + 0.072*"hotel" + 0.053*"stay" + 0.050*"taco" + 0.030*"flight"')
---
(9, '0.015*"car" + 0.014*"help" + 0.011*"work" + 0.011*"store" + 0.010*"need" + 

### Probabilidades dos tópicos em cada documento

In [22]:
docsProbabilities = model_prep.get_document_topics(corpus_prep, minimum_probability=0)

docsProbabilities

In [23]:
#Probabilidade de cada tópico por documento. Tupla (topico, probabilidade)
for docDistribution in docsProbabilities[0:10]:
    print(docDistribution)

[(0, 0.0071519385), (1, 0.15870532), (2, 0.0071519488), (3, 0.2594404), (4, 0.00715397), (5, 0.007151464), (6, 0.0071520037), (7, 0.007152521), (8, 0.007151464), (9, 0.53178895)]
[(0, 0.0016159652), (1, 0.0016160054), (2, 0.0016156166), (3, 0.0016158244), (4, 0.20348512), (5, 0.001615453), (6, 0.14092676), (7, 0.0016159896), (8, 0.0016155088), (9, 0.6442778)]
[(0, 0.0018528984), (1, 0.0018533268), (2, 0.0018526929), (3, 0.0018531341), (4, 0.0018529623), (5, 0.0018524573), (6, 0.0018528637), (7, 0.5233999), (8, 0.0018524555), (9, 0.46177733)]
[(0, 0.0029422392), (1, 0.002942852), (2, 0.0029422129), (3, 0.0029421565), (4, 0.11577043), (5, 0.0029418329), (6, 0.002942348), (7, 0.0029422576), (8, 0.002941841), (9, 0.86069185)]
[(0, 0.08099919), (1, 0.00048701814), (2, 0.00048694856), (3, 0.00048704544), (4, 0.5227979), (5, 0.005358445), (6, 0.029963206), (7, 0.0004870218), (8, 0.00048686867), (9, 0.35844636)]
[(0, 0.33744976), (1, 0.0077031488), (2, 0.00770256), (3, 0.31241146), (4, 0.00770

# Análise de Sentimentos

### Utilizando o VADER

In [24]:
from nltk.sentiment import SentimentIntensityAnalyzer

sent = SentimentIntensityAnalyzer()

print(sent.polarity_scores("Yes! Data mining is very powerful!"))

{'neg': 0.0, 'neu': 0.385, 'pos': 0.615, 'compound': 0.7489}


A pontuação composta (compound) é calculada de tal maneira que representa a polaridade de -1 a 1, onde +1 é o mais positivo e -1 é o mais negativo.

### Utilizando o Textblob

In [30]:
!pip install -U textblob
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 32.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 34.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]


In [26]:
from textblob import TextBlob

frase = TextBlob("Yes! Data mining is very powerful!")
frase.sentiment

Sentiment(polarity=0.48750000000000004, subjectivity=1.0)

### Exemplo com os reviews do Yelp

In [27]:
def verifica_MuitoPositivo(review):
    
    return sent.polarity_scores(review)["compound"] > 0.8


for review in reviewsCharlotte_original[0:10]:
    
    if verifica_MuitoPositivo(review):
        print(review)
        print('----')

I loved the customer service! Fast, but the doctor took her time to make sure I was happy in my contact lenses. Thank you!
----
Robert and his team took great care of me when repairing my car back up camera. The team was able to identify the issue and resolve it for a low cost option. The team could have taken advantage of me and oversold equipment that wasn't needed, however they were very upfront about the root cause of the issue and I was very pleased with the repair and the repair cost !
----
Edit: I'm lowering this review from 5-stars to 3-stars.  I wrote the original review prematurely. After having been around inside my car a bit I have noticed broken rivets & clips and pieces of carpet were never put back where they should have been. Also, many leftover pieces of wire and debris were left in the crevices and sides of the seat. My biggest issue is every time I turn my car on I hear a pop come from the JL subwoofer that was bought and installed at Freeman's. I brought this up to 

# Transformers - Frases similares com sBert

In [32]:
#!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util

model_RobLarge = SentenceTransformer('all-MiniLM-L6-v2')


In [34]:
dfTweetsTrump = pd.read_csv("../data/tweets_trump.csv")

#Usando somente com os 10000 primeiros -
embeddingsTweets_ROBbase = model_RobLarge.encode(dfTweetsTrump.loc[0:10000,'text'], convert_to_tensor=True)

In [35]:
embeddingsTweets_ROBbase
matSimi = cosine_similarity(embeddingsTweets_ROBbase.cpu().detach().numpy())

In [36]:
maior = 0
count = 0
docID = -1
docTarget = 120

for i in matSimi[docTarget,:]:
    if count == docTarget:
        count +=1
        continue
        
    if i> maior:
        maior = i
        docID = count
    
    count+=1

In [37]:
print('****target***')
print(dfTweetsTrump.loc[docTarget,'text'])
print()
print("*** mais similar ***")
print(dfTweetsTrump.loc[docID,'text'])

print(maior)
print(docID)

****target***
RT @Claire_FOX5: #BREAKING:  @GaSecofState's office confirms Floyd County has found 2,600 ballots during audit.  Says Sec. Raffensperger wa…

*** mais similar ***
Investigators Dispatched After Fulton County Discovers ‘Issue‘ with Ballot Reporting https://t.co/lShmKksQ0O via @BreitbartNews
0.61102235
2582


## Outros exemplos com Transformers

### Classificação - zero-shot

In [38]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model='valhalla/distilbart-mnli-12-1')


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/890M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/890M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use mps:0


In [39]:
classifier(
    ["This is the Data Mining course from CPGEI",
     "I have a very good stock option, gonna be rich!"],
    
    candidate_labels=["education", "politics", "finance", "computer", "transport"],
)

[{'sequence': 'This is the Data Mining course from CPGEI',
  'labels': ['computer', 'education', 'transport', 'finance', 'politics'],
  'scores': [0.510725200176239,
   0.3371286392211914,
   0.05968799442052841,
   0.05526547133922577,
   0.0371926873922348]},
 {'sequence': 'I have a very good stock option, gonna be rich!',
  'labels': ['finance', 'transport', 'computer', 'education', 'politics'],
  'scores': [0.8379284739494324,
   0.050801340490579605,
   0.04393551126122475,
   0.04384545609354973,
   0.023489248007535934]}]

### NER - Named Entity Recognition

In [40]:
from transformers import pipeline

ner = pipeline("ner", grouped_entities=True)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use mps:0
/Users/pedro/code/tashima42/ciencia-de-dados/mineracao-dados/venv3.12/lib/python3.12/site-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


In [41]:
ner("I am Thiago and a teach Data Mining at CPGEI.")

[{'entity_group': 'PER',
  'score': 0.99124414,
  'word': 'Thiago',
  'start': 5,
  'end': 11},
 {'entity_group': 'ORG',
  'score': 0.9959862,
  'word': 'CPGEI',
  'start': 39,
  'end': 44}]

In [ ]:
## exemplos de transformers: https://huggingface.co/models

In [ ]:
## spacy transformers: https://spacy.io/usage/embeddings-transformers#transformers